<a href="https://colab.research.google.com/github/dwi-aprilianto/GrandBDC_Case1/blob/main/GrandBDC_Case_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#instal library
!pip install openpyxl scikit-learn -q

In [2]:
#import library
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

In [3]:
#load dataset
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
train_df = pd.read_excel('/content/drive/MyDrive/GrandBDC_Case1/case_1_labeled_data.xlsx')
test_df = pd.read_excel('/content/drive/MyDrive/GrandBDC_Case1/case_1_text_to_predict.xlsx')
template_df = pd.read_excel('/content/drive/MyDrive/GrandBDC_Case1/case_1_template_sheet.xlsx')

Mounted at /content/drive


In [4]:
#preprocessing
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
train_df['full_text'] = train_df['full_text'].apply(clean_text)
test_df['full_text'] = test_df['full_text'].apply(clean_text)

In [5]:
#label endoding
le = LabelEncoder()
train_df['label_encoded'] = le.fit_transform(train_df['label'])

In [6]:
#split data training
X_train, X_val, y_train, y_val = train_test_split(
    train_df['full_text'],
    train_df['label_encoded'],
    test_size=0.2,
    stratify=train_df['label_encoded'],
    random_state=42)

In [7]:
#mengubah teks menjadi angka
#tidak menggunakan indobert/bert karena keterbatasan knowledge dari tim kami sehingga menggunakan TF-IDF yang sudah kami pelajari
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(test_df['full_text'])

In [8]:
#training model menggunakan logistic regression
model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced')
model.fit(X_train_tfidf, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [9]:
#evaluasi model
val_pred = model.predict(X_val_tfidf)
score = balanced_accuracy_score(y_val, val_pred)
print("Balanced Accuracy:", score)

Balanced Accuracy: 0.5963731553570641


In [10]:
#prediksi data testing
test_pred = model.predict(X_test_tfidf)
labels = le.inverse_transform(test_pred)

In [13]:
#save submission prediksi
submission_df = template_df.copy()
submission_df['label'] = labels
submission_df.to_excel('/content/drive/MyDrive/GrandBDC_Case1/submission_case1.xlsx', index=False)
print("Dataset tersimpan: submission_case1.xlsx")

Dataset tersimpan: submission_case1.xlsx


Dokumentasi:

```
https://drive.google.com/drive/folders/1xclZZEO88LfypJvVLyNrxOpT6dSckTdE?usp=sharing
```

